In [15]:
import gzip
path = "./amazon_reviews_us_Musical_Instruments_v1_00.tsv.gz"
f = gzip.open(path, 'rt', encoding="utf8")

header = f.readline()
header = header.strip().split('\t')

review_dataset = []

pairsSeen = set()

for line in f:
    fields = line.strip().split('\t')
    d = dict(zip(header, fields))
    ui = (d['customer_id'], d['product_id'])
    if ui in pairsSeen:
        print("Skipping duplicate user/item:", ui)
        continue
    pairsSeen.add(ui)
    d['star_rating'] = int(d['star_rating'])
    d['helpful_votes'] = int(d['helpful_votes'])
    d['total_votes'] = int(d['total_votes'])
    review_dataset.append(d)

reviewDataTrain = review_dataset[:int(len(review_dataset)*0.9)]
reviewDataTest = review_dataset[int(len(review_dataset)*0.9):]

Skipping duplicate user/item: ('46953315', 'B00QM3CNN6')
Skipping duplicate user/item: ('31616428', 'B0026RB0G8')
Skipping duplicate user/item: ('47240912', 'B008I653SC')
Skipping duplicate user/item: ('14503091', 'B003FRMRC4')
Skipping duplicate user/item: ('38538360', 'B00HVLUR86')
Skipping duplicate user/item: ('43448024', 'B00HVLUR86')
Skipping duplicate user/item: ('51525270', 'B00HVLUR86')
Skipping duplicate user/item: ('20652160', 'B004OU2IQG')
Skipping duplicate user/item: ('10964440', 'B00HVLUR86')
Skipping duplicate user/item: ('20043677', 'B00HVLUR86')
Skipping duplicate user/item: ('44796499', 'B00HVLUSGM')
Skipping duplicate user/item: ('29066899', 'B0002CZSYO')
Skipping duplicate user/item: ('10385056', 'B004OU2IQG')
Skipping duplicate user/item: ('1658551', 'B00HVLURL8')
Skipping duplicate user/item: ('907433', 'B00N9Q2E5G')
Skipping duplicate user/item: ('39412969', 'B00HVLUR86')
Skipping duplicate user/item: ('4901688', 'B00HVLUR86')
Skipping duplicate user/item: ('234

In [48]:
import numpy as np
from collections import defaultdict
usersPerItem = defaultdict(set) # Maps an item to the users who rated it
itemsPerUser = defaultdict(set) # Maps a user to the items that they rated
itemNames = {}
ratingDict = {} # To retrieve a rating for a specific user/item pair
reviewsPerUser = defaultdict(list)

for d in reviewDataTrain:
    user,item = d['customer_id'], d['product_id']
    usersPerItem[item].add(user)
    itemsPerUser[user].add(item)
    reviewsPerUser[user].append(d)

for d in review_dataset:
    user,item = d['customer_id'], d['product_id']
    ratingDict[(user,item)] = d['star_rating']
    itemNames[item] = d['product_title']
def Jaccard(s1, s2):
    # Implement |s1 & s2|/ |s1 V s2|
    common = len(s1 & s2)
    both = len(s1 | s2)
    try:
        result = common/both
    except:
        print("sets s1 and s2 empty")
        result = 0
    return result

def mostSimilar(i, N, usersPerItem):
    # initalize list to store similarity results (similarity, itemID)
    similarities = []
    s1 = usersPerItem[i]
    for item in usersPerItem.keys():
        s2 = usersPerItem[item]
        if s1 == s2:
            continue
        jac = Jaccard(s1,s2)
        similarities.append((jac,item))
    # sort similarites
    similarities.sort(reverse=True)
    return(similarities[:N])

def MSE(y, ypred):
    y_test = np.array(y)
    y_pred = np.array(ypred)
    mse = np.mean((y_test - y_pred)**2)
    return(mse)

def getMeanRating(dataTrain):
    mean_rating = np.mean(np.array([d['star_rating'] for d in dataTrain]))
    return(mean_rating)

def getUserAverages(itemsPerUser, ratingDict):
    # Implement (should return a dictionary mapping users to their averages)
    #return userAverages
    userAverages = {}
    for user in itemsPerUser.keys():
        ratings = [ratingDict[(user, item)] for item in itemsPerUser[user]]
        userAverages[user] = np.mean(ratings)
    return(userAverages)

def getItemAverages(usersPerItem, ratingDict):
    # Implement...
    itemAverages = {}
    for item in usersPerItem.keys():
        ratings = [ratingDict[(user, item)] for user in usersPerItem[item]]
        itemAverages[item] = np.mean(ratings)
    return(itemAverages)
ratingMean = getMeanRating(reviewDataTrain)

userAverages = getUserAverages(itemsPerUser, ratingDict)

itemAverages = getItemAverages(usersPerItem, ratingDict)
def predictRating(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    """
    r(u, i) = R_i_bar + [ SUM(j in I_u) ( (R_u,j - R_j_bar) * Sim(i, j) ) ] / 
                      [ SUM(j in I_u) ( Sim(i, j) ) ]
    """
    ratings = []
    sims = []
    # loop over reviews by user user_review_i
    for user_review_i in reviewsPerUser[user]:
        item_id = user_review_i["product_id"]
        # don't include comparision of item against itself
        if item_id == item:
            continue
        # calc (R_u,j - R_j_bar)
        rating_diff = user_review_i['star_rating'] - itemAverages[item_id]
        ratings.append(rating_diff)
        # calc Sim(i, j)
        sim = Jaccard(usersPerItem[item],usersPerItem[item_id])
        sims.append(sim)

    # if no similar items then return item avg
    if len(sims) == 0: 
        final_rating = itemAverages.get(item,ratingMean)
        return(final_rating)
    # calc sums for non empty case
    else:
        ratings_arr = np.array(ratings)
        sims_arr = np.array(sims)
        sims_sum = np.sum(sims_arr)
        # can't divide by zero
        if sims_sum >0:
            final_rating = itemAverages.get(item,ratingMean) + np.dot(ratings_arr, sims_arr)/sims_sum
            return(final_rating)
        else:
            final_rating = itemAverages.get(item,ratingMean)
            return(final_rating)

def predictRatingQ7(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    k=0
    mu = ratingMean
    
    # calc user bias
    num_user_reviews = len(reviewsPerUser.get(user, []))
    user_avg = userAverages.get(user, mu)
    b_u_raw = user_avg - mu
    b_u = b_u_raw * (num_user_reviews / (num_user_reviews + k))
    
    # calc item bias
    num_item_raters = len(usersPerItem.get(item, set()))
    item_avg = itemAverages.get(item, mu)
    b_i_raw = item_avg - mu
    b_i = b_i_raw * (num_item_raters / (num_item_raters + k))
    
    # calc final rating
    final_rating = mu + b_u + b_i
    
    return(final_rating)

In [49]:
import numpy as np

def testQ6():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    q7Predictions = [predictRatingQ7(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]

    labels = [d['star_rating'] for d in reviewDataTest]
    
    m1 = MSE(simPredictions, labels)
    m2 = MSE(q7Predictions, labels)
    m3 = MSE(alwaysPredictMean, labels)

    print(f"sim pred mse: {m1}")
    print(f"q7 pred mse: {m2}")
    print(f"just mse: {m3}")
    #print(q7Predictions[:5])
    #print(labels[:5])
    
    # Autograder checks the MSE of your predictions and (some of) the simPrediction values

testQ6()

ZeroDivisionError: division by zero

In [35]:
import numpy as np
from itertools import combinations_with_replacement

def hodges_lehmann_one_sample(data):
    """
    Calculates the Hodges-Lehmann estimator for a one-sample scenario.

    Args:
        data (list or numpy.ndarray): A numeric vector of data values.

    Returns:
        float: The Hodges-Lehmann estimator (pseudo-median).
    """
    if not isinstance(data, (list, np.ndarray)):
        raise TypeError("Input 'data' must be a list or numpy array.")
    if len(data) == 0:
        raise ValueError("Input 'data' cannot be empty.")

    # Calculate all possible Walsh averages
    walsh_averages = []
    for i, j in combinations_with_replacement(range(len(data)), 2):
        walsh_averages.append((data[i] + data[j]) / 2)

    # The Hodges-Lehmann estimator is the median of these averages
    return np.median(walsh_averages)

# Example usage:
data_sample = [1, 3, 7, 8, 2, 5]
hl_estimate = hodges_lehmann_one_sample(data_sample)
print(f"The Hodges-Lehmann estimator for the sample is: {hl_estimate}")
print(f"mean: {np.mean(data_sample)}")

The Hodges-Lehmann estimator for the sample is: 4.5
mean: 4.333333333333333


In [ ]:
import numpy as np
ratingMean = getMeanRating(reviewDataTrain)

userAverages = getUserAverages(itemsPerUser, ratingDict)

itemAverages = getItemAverages(usersPerItem, ratingDict)
def predictRatingQ7(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    # Your solution here
    ratings = []
    sims = []
    def Jaccard(s1, s2):
        # Implement |s1 & s2|/ |s1 V s2|
        common = len(s1 & s2)
        both = len(s1 | s2)
        try:
            result = common/both
        except:
            print("sets s1 and s2 empty")
            result = 0
        return result
    # loop over reviews by user user_review_i
    love_count = 0
    hate_count = 0
    for user_review_i in reviewsPerUser[user]:
        item_id = user_review_i["product_id"]
        if 'love' in user_review_i['review_headline']:
            love_count +=1
        if "don't" in user_review_i['review_headline']:
            hate_count +=1
        # don't include comparision of item against itself
        if item_id == item:
            continue
        # calc (R_u,j - R_j_bar)
        rating_diff = user_review_i['star_rating'] - itemAverages[item_id]
        ratings.append(rating_diff)
        # calc Sim(i, j)
        sim = Jaccard(usersPerItem[item],usersPerItem[item_id])
        sims.append(sim)
    
    # if no similar items then return item avg
    if len(sims) == 0: 
        final_rating = itemAverages.get(item,ratingMean)
        return(final_rating)
    # calc sums for non empty case
    else:
        ratings_arr = np.array(ratings)
        sims_arr = np.array(sims)
        sims_sum = np.sum(sims_arr)
        # can't divide by zero
        if sims_sum >0:
            final_rating = itemAverages.get(item,ratingMean) + np.dot(ratings_arr, sims_arr)/sims_sum
            return(final_rating)
        else:
            final_rating = itemAverages.get(item,ratingMean)
            return(final_rating)
def MSE(y, ypred):
    y_test = np.array(y)
    y_pred = np.array(ypred)
    mse = np.mean((y_test - y_pred)**2)
    return(mse)
def testQ6():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    q7Predictions = [predictRatingQ7(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]

    labels = [d['star_rating'] for d in reviewDataTest]
    
    m1 = MSE(simPredictions, labels)
    m2 = MSE(q7Predictions, labels)
    m3 = MSE(alwaysPredictMean, labels)

    print(f"sim pred mse: {m1}")
    print(f"q7 pred mse: {m2}")
    print(f"just mse: {m3}")
    #print(q7Predictions[:5])
    #print(labels[:5])
    
    # Autograder checks the MSE of your predictions and (some of) the simPrediction values

testQ6()

sim pred mse: nan
q7 pred mse: nan
just mse: 1.6236571809194924


In [12]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# --- Assume 'df' is your DataFrame loaded with the data ---
df = pd.DataFrame(reviewDataTrain)

def get_top_n_words(headline_series, n=10):
    """
    A helper function to be applied to each group.
    It takes a pandas Series of headlines, counts all words,
    and returns the top N most frequent.
    """
    # Ensure all data is string and handle potential empty/NaN headlines
    headlines = headline_series.fillna('')
    
    # Don't run on an empty group
    if headlines.empty:
        return []

    # Initialize the vectorizer to count words and remove English stop words
    # We set max_features to 2000 to be efficient, as we only need the top.
    vectorizer = CountVectorizer(stop_words='english', max_features=2000)
    
    try:
        # Create the word-count matrix
        X = vectorizer.fit_transform(headlines)
    except ValueError:
        # This can happen if all headlines in a group were just stop words
        return []

    # Sum the counts of each word across all headlines in the group
    # X.sum(axis=0) gives a (1, num_words) matrix
    word_counts = np.asarray(X.sum(axis=0)).ravel()
    
    # Get the actual words (features)
    feature_names = vectorizer.get_feature_names_out()
    
    # Get the indices that would sort the counts in descending order
    top_n_indices = word_counts.argsort()[::-1][:n]
    
    # Map the indices back to the word names
    top_words = [feature_names[i] for i in top_n_indices]
    
    return top_words

# --- Main Execution ---

# 1. Group by 'star_rating', select the 'review_headline', and apply our function
top_words_series = df.groupby('star_rating')['review_headline'].apply(get_top_n_words)

# 2. Convert the resulting pandas Series into the dictionary you want
top_words_by_rating = top_words_series.to_dict()

# --- Display the result ---
import json
print(json.dumps(top_words_by_rating, indent=2))

{
  "1": [
    "star",
    "buy",
    "don",
    "work",
    "quality",
    "money",
    "product",
    "good",
    "poor",
    "bad"
  ],
  "2": [
    "stars",
    "good",
    "quality",
    "great",
    "sound",
    "work",
    "poor",
    "cheap",
    "like",
    "bad"
  ],
  "3": [
    "stars",
    "good",
    "great",
    "ok",
    "price",
    "works",
    "sound",
    "nice",
    "quality",
    "bad"
  ],
  "4": [
    "good",
    "great",
    "stars",
    "price",
    "nice",
    "works",
    "sound",
    "guitar",
    "product",
    "quality"
  ],
  "5": [
    "stars",
    "great",
    "good",
    "price",
    "product",
    "guitar",
    "love",
    "nice",
    "best",
    "works"
  ]
}


In [13]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Assume 'df' is your DataFrame loaded with the data ---
# df = pd.DataFrame(your_list_of_dicts)

# --- 1. Consolidate Headlines into Documents ---
# We create 5 "documents," one for each star rating,
# by joining all headlines for that rating into a single string.

# First, handle potential NaN headlines
df['review_headline'] = df['review_headline'].fillna('')

# Group by star rating and join all headlines
grouped_headlines = df.groupby('star_rating')['review_headline'].apply(' '.join)

# --- 2. Apply TF-IDF Vectorizer ---
# We'll treat each star rating's combined text as a separate document.
corpus = grouped_headlines.tolist()
ratings_index = grouped_headlines.index # This holds the star ratings [1, 2, 3, 4, 5]

# Initialize TF-IDF
# This will calculate the TF-IDF score for each word *per star rating*
vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)

# Fit and transform the corpus
tfidf_matrix = vectorizer.fit_transform(corpus)

# Get the actual word names
feature_names = vectorizer.get_feature_names_out()

# --- 3. Extract Top 10 Words for Each Rating ---
top_characteristic_words = {}

# Loop through each row of the TF-IDF matrix (each row is a star rating)
for i, rating in enumerate(ratings_index):
    
    # Get the scores for all words for this specific star rating
    row = tfidf_matrix.getrow(i).toarray().ravel()
    
    # Get the indices of the top 10 scores in descending order
    top_n_indices = row.argsort()[::-1][:10]
    
    # Map those indices back to the word names
    top_words = [feature_names[idx] for idx in top_n_indices]
    
    # Store in our dictionary
    top_characteristic_words[rating] = top_words

# --- Display the result ---
import json
print(json.dumps(top_characteristic_words, indent=2))

{
  "1": [
    "star",
    "buy",
    "don",
    "work",
    "quality",
    "money",
    "product",
    "good",
    "poor",
    "bad"
  ],
  "2": [
    "stars",
    "good",
    "quality",
    "great",
    "sound",
    "work",
    "poor",
    "cheap",
    "like",
    "bad"
  ],
  "3": [
    "stars",
    "good",
    "great",
    "ok",
    "price",
    "works",
    "sound",
    "nice",
    "quality",
    "bad"
  ],
  "4": [
    "good",
    "great",
    "stars",
    "price",
    "nice",
    "works",
    "sound",
    "guitar",
    "product",
    "quality"
  ],
  "5": [
    "stars",
    "great",
    "good",
    "price",
    "product",
    "guitar",
    "love",
    "nice",
    "best",
    "works"
  ]
}
